# Why Does Training Work? A Hands-On Tour of Generalization

**Case study: a Pokemon-vs-Digimon threshold classifier.**

We want to tell **Pokemon** from **Digimon** using a single number: an *edge-count feature* `e` (Digimon line art is busier, so it has more edges). This tiny problem is a complete instance of the **three-step ML recipe**:

1. **A model with unknown parameters** — a family of candidate functions $f_h$: predict *Digimon* if `e >= h`, else *Pokemon*. The unknown parameter is the threshold `h`.
2. **A loss** — the 0-1 error rate $L(h, D)$: the fraction of examples the threshold gets wrong.
3. **Optimization** — search the hypothesis set $H = \{1, \ldots, 10000\}$ for the `h` with the smallest loss.

The whole notebook circles one question:

> **We only ever train on a small sample. Why should low error on that sample tell us anything about the real world?**

**Roadmap.** We will (1) build a calibrated synthetic universe, (2) define the model/loss and find the ideal threshold, (3) watch the gap between *training* loss and *ideal* loss open up under i.i.d. sampling, (4) formalize "good vs. bad" training sets and estimate $P(\text{bad})$ by simulation, (5) derive and verify the **Hoeffding + union bound**, (6) invert it for **sample complexity**, and (7) explore the **$|H|$ complexity tradeoff** — closing with the teaser that resolves it: *deep learning*.

Each section below maps to one learning objective; every interactive widget appears right after the static version of the same idea, so you first *see* the result, then *manipulate* it.

In [ ]:
# Setup: imports, reproducibility, and plotting defaults.
import numpy as np
import matplotlib.pyplot as plt

# ipywidgets is optional. Guard the import so "Run All" never aborts on a runtime
# without widgets; interactive cells (C08/C12/C18/C23) fall back to a static plot.
try:
    import ipywidgets as widgets
    from ipywidgets import interact, IntSlider, FloatSlider, SelectionSlider, fixed
    WIDGETS_AVAILABLE = True
except ImportError:
    widgets = None
    WIDGETS_AVAILABLE = False
    print("WARNING: ipywidgets not available -> interactive cells render a single static plot.")

# Single shared RNG, used ONLY for one-shot data construction in C04. Do not reseed.
GLOBAL_SEED = 42
rng = np.random.default_rng(GLOBAL_SEED)

# Consistent plot styling across the whole notebook.
plot_defaults = {
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "lines.linewidth": 2,
}
plt.rcParams.update(plot_defaults)

# Sanity checks
assert np.__version__ is not None and callable(plt.plot)
assert isinstance(GLOBAL_SEED, int) and hasattr(rng, "normal")
print(f"ipywidgets available: {WIDGETS_AVAILABLE}")

## From images to one number: the edge-count feature `e`

Training a real image classifier would drag in convolutional networks, GPUs, and two image datasets we would have to ship. None of that is needed to study **generalization** — the phenomenon we actually care about here. So we compress each creature into a single scalar:

- **`e` = edge count.** Digimon artwork is line-complex (mechanical detail, sharp contours) → **more edges**. Pokemon are rounder and simpler → **fewer edges**.

This gives a clean **1-D feature** on which a *threshold* is the natural classifier: small `e` looks like Pokemon, large `e` looks like Digimon. Because the two creature types overlap in how detailed they are, their feature distributions **overlap** — so no single threshold is perfect, and some irreducible error is unavoidable. That overlap is exactly what makes the generalization story interesting.

We generate `e` synthetically from two calibrated distributions so the notebook is **fully self-contained** (no downloads) while still reproducing the lecture's numbers: an optimal threshold near $h_{\text{all}} \approx 4824$ with ideal loss $L(h_{\text{all}}, D_{\text{all}}) \approx 0.28$.

In [ ]:
# Build the ideal universe D_all from two calibrated overlapping Gaussians.
# Label convention (fixed everywhere): 0 = Pokemon (fewer edges), 1 = Digimon (more edges).
N_pokemon, N_digimon = 819, 971
N_all = N_pokemon + N_digimon  # 1790

# Calibration constants tuned to the slides (h_all ~ 4824, L(h_all, D_all) ~ 0.28).
# Digimon edge counts are both higher-centered and more spread out (larger sigma),
# which lands the optimal threshold near 4824 while keeping the Bayes error ~0.28.
mu_pokemon, mu_digimon = 4050.0, 5250.0
sigma_pokemon, sigma_digimon = 900.0, 1300.0

# Draw synthetic edge counts from the shared rng, clip into the hypothesis range
# [1, 10000], and cast to int so threshold comparisons (e >= h) are exact.
e_pokemon = np.clip(np.round(rng.normal(mu_pokemon, sigma_pokemon, N_pokemon)), 1, 10000).astype(int)
e_digimon = np.clip(np.round(rng.normal(mu_digimon, sigma_digimon, N_digimon)), 1, 10000).astype(int)

e_all = np.concatenate([e_pokemon, e_digimon])
y_all = np.concatenate([np.zeros(N_pokemon, int), np.ones(N_digimon, int)])
D_all = {"e": e_all, "y": y_all}

# Calibration checks (the exact optimum/loss are computed in C09).
assert e_all.shape == (1790,) and y_all.shape == (1790,)
assert e_all.min() >= 1 and e_all.max() <= 10000
assert set(np.unique(y_all).tolist()) == {0, 1} and int(y_all.sum()) == 971
err = float(np.mean((e_all >= 4824).astype(int) != y_all))
assert 0.24 <= err <= 0.32, f"calibration off: {err:.3f} (nudge mu_digimon +/-100 or sigma +/-50)"
print(f"Built D_all: {N_all} samples ({N_pokemon} Pokemon + {N_digimon} Digimon). "
      f"L(h=4824, D_all) = {err:.3f}")

In [ ]:
# Static reference plot: the two overlapping edge-count histograms.
bins = np.linspace(1, 10000, 61)  # shared bin edges so the visual comparison is valid

assert e_pokemon.size > 0 and e_digimon.size > 0

fig, ax = plt.subplots()
ax.hist(e_pokemon, bins=bins, alpha=0.55, label="Pokemon (y=0)", color="tab:blue")
ax.hist(e_digimon, bins=bins, alpha=0.55, label="Digimon (y=1)", color="tab:red")
ax.axvspan(4000, 5400, color="gray", alpha=0.12)
ax.text(4700, ax.get_ylim()[1] * 0.9, "overlap", ha="center", color="dimgray")
ax.set_xlabel("edge-count feature e")
ax.set_ylabel("count")
ax.set_title("Overlapping edge-count distributions - no threshold separates them perfectly")
ax.legend()
histogram_figure = fig
plt.show()

## The model, the hypothesis set, and model complexity

**Model.** For a threshold `h`, the classifier is

$$ f_h(e) = \begin{cases} \text{Digimon } (1) & \text{if } e \ge h \\[2pt] \text{Pokemon } (0) & \text{if } e < h \end{cases} $$

**Hypothesis set.** Letting `h` range over the integers gives a whole family of candidate functions:

$$ H = \{\, f_h : h \in \{1, 2, \ldots, 10000\} \,\}, \qquad |H| = 10000. $$

**Loss.** We score a hypothesis by its **0-1 error rate** on a dataset `D`:

$$ L(h, D) = \frac{1}{|D|} \sum_{(e_n, y_n) \in D} \mathbb{1}\!\big[\, f_h(e_n) \ne y_n \,\big] \;\in\; [0, 1]. $$

**Model complexity = $|H|$.** Here complexity is literally *how many candidate functions we let ourselves choose from*. A bigger $|H|$ can fit the data better — but, as we will see, it also makes it easier to be fooled by a lucky sample. Keep $|H| = 10000$ in mind: it is the knob in every bound that follows. (Cross-entropy is the smooth surrogate you would use for gradient training; here the 0-1 loss is what the theory bounds.)

In [ ]:
# Core reusable model, 0-1 loss, hypothesis grid, and a fast loss-over-grid scan.
H_grid = np.arange(1, 10001)  # literal H = {1, ..., 10000}, |H| = 10000

def predict(e, h):
    """Threshold classifier: 1 (Digimon) if e >= h else 0 (Pokemon). Broadcasts over arrays."""
    return (np.asarray(e) >= h).astype(int)

def loss_01(h, D):
    """0-1 error rate L(h, D) in [0, 1] for a single threshold h on dataset dict D."""
    e, y = D["e"], D["y"]
    return float(np.mean(predict(e, h) != y))

def losses_over_H(H, e, y):
    """Vectorized L(h, .) for every h in H (H must be sorted ascending).

    For threshold h: a Pokemon (y=0) errs iff e >= h; a Digimon (y=1) errs iff e < h.
    Both counts come from searchsorted on per-class sorted arrays, giving
    O(N log N + |H| log N) instead of an O(|H|*N) boolean matrix. Returns a float
    array aligned with H, so losses_over_H(H_grid, D['e'], D['y'])[k] == loss_01(H_grid[k], D).
    """
    e = np.asarray(e); y = np.asarray(y)
    N = e.size
    assert N > 0, "empty dataset"
    pos = np.sort(e[y == 1])  # Digimon edge counts
    neg = np.sort(e[y == 0])  # Pokemon edge counts
    pokemon_err = neg.size - np.searchsorted(neg, H, side="left")  # neg with e >= h
    digimon_err = np.searchsorted(pos, H, side="left")             # pos with e <  h
    return (pokemon_err + digimon_err) / N

# Checks
assert H_grid[0] == 1 and H_grid[-1] == 10000 and H_grid.size == 10000
_L = losses_over_H(H_grid, D_all["e"], D_all["y"])
assert all(abs(_L[h - 1] - loss_01(h, D_all)) < 1e-12 for h in [1, 2500, 4824, 7500, 10000])
assert (predict(D_all["e"], 1) == 1).all()        # h=1 -> everything predicted Digimon
assert 0.0 <= loss_01(4824, D_all) <= 1.0
print("Model, 0-1 loss, and vectorized H-scan ready.")

In [ ]:
# Interactive threshold explorer: drag h to see the decision boundary and live loss.
if "bins" not in globals():
    bins = np.linspace(1, 10000, 61)

def show_threshold(h):
    fig, ax = plt.subplots()
    ax.hist(e_pokemon, bins=bins, alpha=0.55, label="Pokemon", color="tab:blue")
    ax.hist(e_digimon, bins=bins, alpha=0.55, label="Digimon", color="tab:red")
    ax.axvline(h, color="k", ls="--", lw=2, label=f"threshold h={h}")
    L = loss_01(h, D_all)
    ax.set_title(f"h={h}  ->  L(h, D_all) = {L:.3f}")
    ax.set_xlabel("edge-count e")
    ax.set_ylabel("count")
    ax.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    threshold_explorer_widget = interact(
        show_threshold,
        h=IntSlider(min=1, max=10000, step=50, value=4824, continuous_update=False),
    )
else:
    show_threshold(4824)  # static fallback at the optimum
    threshold_explorer_widget = None

In [ ]:
# Optimization over the whole universe: scan all of H, find h_all, CACHE the curve.
L_all_over_H = losses_over_H(H_grid, D_all["e"], D_all["y"])  # cached; reused read-only by C12/C14/C17/C22
loss_curve_all = L_all_over_H  # alias for the structure-named output

i_star = int(np.argmin(L_all_over_H))
h_all = int(H_grid[i_star])
L_h_all = float(L_all_over_H[i_star])

fig, ax = plt.subplots()
ax.plot(H_grid, L_all_over_H, color="tab:purple")
ax.axvline(h_all, color="r", ls="--")
ax.scatter([h_all], [L_h_all], color="r", zorder=5, label=f"h_all={h_all}, L={L_h_all:.3f}")
ax.set_xlabel("threshold h")
ax.set_ylabel("L(h, D_all)")
ax.set_title("Loss over the hypothesis set - optimum h_all")
ax.legend()
plt.show()

assert L_all_over_H.shape == (10000,)
assert 4700 <= h_all <= 4950, f"h_all={h_all} outside slide tolerance - recalibrate C04"
assert 0.26 <= L_h_all <= 0.30, f"L_h_all={L_h_all:.3f} outside slide tolerance - recalibrate C04"
assert L_h_all == float(L_all_over_H.min())
print(f"h_all = {h_all},  L(h_all, D_all) = {L_h_all:.3f}  (the ideal benchmark)")

## The catch: we optimize on `D_train`, not on `D_all`

In `C09` we found $h_{\text{all}}$ by scanning over the **entire universe** $D_{\text{all}}$. But in real life we **never see the whole universe** — we see a finite, randomly drawn **training sample** $D_{\text{train}}$ of size `N`, drawn **i.i.d.** from the same distribution. So what we *actually* do is

$$ h_{\text{train}} = \arg\min_{h \in H} L(h, D_{\text{train}}). $$

This threshold minimizes loss **on the sample**, which is not the same as minimizing loss on the world. Two things can happen:

- **Lucky sample:** $D_{\text{train}}$ resembles the universe, so $h_{\text{train}} \approx h_{\text{all}}$ and the sample loss is an honest estimate.
- **Unlucky sample:** $D_{\text{train}}$ happens to look cleanly separable in a way the universe is not, so $h_{\text{train}}$ earns a *suspiciously low training loss* but a *high* loss on $D_{\text{all}}$. That gap is **overfitting**.

The quantity we care about is the **generalization gap**: how far $L(h_{\text{train}}, D_{\text{all}})$ sits above the best we could do, $L(h_{\text{all}}, D_{\text{all}})$. Next we make it appear.

In [ ]:
# Worked lucky/unlucky example: minimize on a small i.i.d. D_train, compare to ideal.
N_demo = 30  # small N makes overfitting visible

def draw_train(seed, N):
    """Draw an i.i.d. training sample (with replacement) from D_all using a local RNG."""
    g = np.random.default_rng(seed)
    idx = g.choice(N_all, size=N, replace=True)
    return {"e": e_all[idx], "y": y_all[idx]}

records = []
for seed in range(0, 50):
    D_tr = draw_train(seed, N_demo)
    Lt = losses_over_H(H_grid, D_tr["e"], D_tr["y"])
    h_tr = int(H_grid[np.argmin(Lt)])
    L_tr_train = float(Lt.min())
    L_tr_all = float(L_all_over_H[h_tr - 1])  # index by h-1 because H starts at 1
    records.append({"seed": seed, "h_train": h_tr,
                    "L_train_on_train": L_tr_train, "L_train_on_all": L_tr_all,
                    "gap": L_tr_all - L_tr_train})

lucky = min(records, key=lambda r: abs(r["L_train_on_all"] - L_h_all))   # h_train near h_all
unlucky = max(records, key=lambda r: r["gap"])                          # low train loss, high ideal loss

# Primary outputs come from the lucky case.
D_train = draw_train(lucky["seed"], N_demo)
h_train = lucky["h_train"]
L_train_on_train = lucky["L_train_on_train"]
L_train_on_all = lucky["L_train_on_all"]
generalization_gap_example = {"N": N_demo, "lucky": lucky, "unlucky": unlucky, "L_h_all": L_h_all}

assert 1 <= h_train <= 10000
assert L_train_on_train <= L_train_on_all + 1e-9   # training loss is optimistic vs ideal
assert generalization_gap_example["unlucky"]["gap"] >= generalization_gap_example["lucky"]["gap"]
assert abs(L_train_on_all - L_all_over_H[h_train - 1]) < 1e-12

print(f"Ideal:    h_all={h_all}, L(h_all, D_all)={L_h_all:.3f}")
print(f"Lucky   (seed {lucky['seed']:>2}): h_train={lucky['h_train']:>5}, "
      f"train L={lucky['L_train_on_train']:.3f}, ideal L={lucky['L_train_on_all']:.3f}, gap={lucky['gap']:.3f}")
print(f"Unlucky (seed {unlucky['seed']:>2}): h_train={unlucky['h_train']:>5}, "
      f"train L={unlucky['L_train_on_train']:.3f}, ideal L={unlucky['L_train_on_all']:.3f}, gap={unlucky['gap']:.3f}")

In [ ]:
# Interactive sampling experiment: watch the train/ideal gap shrink (in variance) with N.
def explore_sampling(N, seed):
    g = np.random.default_rng(seed)
    idx = g.choice(N_all, size=N, replace=True)
    et, yt = e_all[idx], y_all[idx]
    Lt = losses_over_H(H_grid, et, yt)
    h_tr = int(H_grid[np.argmin(Lt)])
    gap = float(L_all_over_H[h_tr - 1] - Lt.min())
    fig, ax = plt.subplots()
    ax.plot(H_grid, L_all_over_H, label="L(h, D_all)", color="tab:gray")
    ax.plot(H_grid, Lt, label=f"L(h, D_train) N={N}", color="tab:green", alpha=0.8)
    ax.axvline(h_all, color="k", ls=":", label=f"h_all={h_all}")
    ax.axvline(h_tr, color="tab:green", ls="--", label=f"h_train={h_tr}")
    ax.set_title(f"N={N}, seed={seed} | train L={Lt.min():.3f}, "
                 f"ideal L(h_train)={L_all_over_H[h_tr - 1]:.3f}, gap={gap:.3f}")
    ax.set_xlabel("threshold h")
    ax.set_ylabel("loss")
    ax.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    sampling_experiment_widget = interact(
        explore_sampling,
        N=IntSlider(min=20, max=1000, step=20, value=50, continuous_update=False),
        seed=IntSlider(min=0, max=50, step=1, value=0, continuous_update=False),
    )
else:
    explore_sampling(50, 0)  # static fallback
    sampling_experiment_widget = None

## Reframing it probabilistically: "good" vs. "bad" training sets

We cannot control which sample we draw, so let us reason about the **probability** of drawing a misleading one. Fix a tolerance $\epsilon > 0$.

> A training set $D_{\text{train}}$ is **good** if it estimates the loss of **every** hypothesis well:
> $$ \forall\, h \in H : \quad \big| L(h, D_{\text{train}}) - L(h, D_{\text{all}}) \big| \le \epsilon. $$
> Otherwise it is **bad** (some `h` is badly mis-estimated).

Note the **"for all `h`"** — a good set is good *uniformly* across the whole hypothesis set, not just at one threshold. This strengthening is exactly what forces the union bound later.

**Why "good" is enough.** If $D_{\text{train}}$ is good, then for the threshold we actually pick,
$$ L(h_{\text{train}}, D_{\text{all}}) - L(h_{\text{all}}, D_{\text{all}}) \le 2\epsilon. $$

*Sketch:* $L(h_{\text{train}}, D_{\text{all}}) \le L(h_{\text{train}}, D_{\text{train}}) + \epsilon$ (good set) $\le L(h_{\text{all}}, D_{\text{train}}) + \epsilon$ ($h_{\text{train}}$ is optimal on the sample) $\le L(h_{\text{all}}, D_{\text{all}}) + 2\epsilon$ (good set again). So **a good sample guarantees we generalize to within $2\epsilon$** — and our whole job collapses to making $P(D_{\text{train}}\text{ is bad})$ small.

In [ ]:
# Good/bad-sample predicate over the FULL hypothesis set (reuses cached L_all_over_H).
assert "L_all_over_H" in globals() and L_all_over_H.shape == (10000,)

def max_gap_over_H(et, yt):
    """Largest |L(h, D_train) - L(h, D_all)| over all h in H (recomputes only the train curve)."""
    et = np.asarray(et); yt = np.asarray(yt)
    assert et.shape == yt.shape
    Lt = losses_over_H(H_grid, et, yt)
    return float(np.max(np.abs(Lt - L_all_over_H)))

def is_bad_sample(et, yt, epsilon):
    """Bad iff SOME h violates |L(h,D_train) - L(h,D_all)| <= epsilon (negation of for-all-h)."""
    return bool(max_gap_over_H(et, yt) > epsilon)

assert max_gap_over_H(e_all, y_all) == 0.0 and is_bad_sample(e_all, y_all, 0.01) is False
assert isinstance(is_bad_sample(e_all[:10], y_all[:10], 0.05), bool)
assert max_gap_over_H(e_all[:50], y_all[:50]) >= 0.0
print("Good/bad predicate ready (bad = some h breaks the epsilon gap).")

In [ ]:
# Monte Carlo estimate of P(D_train is bad).
def monte_carlo_p_bad(N, epsilon, n_trials=500, seed=0):
    assert n_trials >= 1 and N >= 1
    g = np.random.default_rng(seed)  # one local generator across all trials -> reproducible
    bad = 0
    for _ in range(n_trials):
        idx = g.choice(N_all, size=N, replace=True)
        bad += int(is_bad_sample(e_all[idx], y_all[idx], epsilon))
    return bad / n_trials

# Representative point matching the slides.
N_mc = 100
epsilon_mc = 0.1
monte_carlo_trials = 500
p_bad_empirical = monte_carlo_p_bad(N_mc, epsilon_mc, monte_carlo_trials, seed=0)

assert 0.0 <= p_bad_empirical <= 1.0
print(f"Empirical P(D_train is bad) at N={N_mc}, epsilon={epsilon_mc}, "
      f"{monte_carlo_trials} trials: {p_bad_empirical:.3f}")
print(f"(Monte Carlo noise ~ sqrt(p(1-p)/n_trials) = "
      f"{(p_bad_empirical*(1-p_bad_empirical)/monte_carlo_trials)**0.5:.3f})")

## The centerpiece: Hoeffding + a union bound

**Step 1 — one fixed hypothesis (Hoeffding).** For a *single* `h`, $L(h, D_{\text{train}})$ is the mean of `N` i.i.d. 0-1 terms whose expectation is $L(h, D_{\text{all}})$. Hoeffding's inequality for variables in $[0, 1]$ gives

$$ P\big(\, |L(h, D_{\text{train}}) - L(h, D_{\text{all}})| > \epsilon \,\big) \le 2\,e^{-2 N \epsilon^2}. $$

This is the probability that **this one** hypothesis is badly estimated. It shrinks **exponentially in `N`** and in $\epsilon^2$.

**Step 2 — all hypotheses at once (union bound).** $D_{\text{train}}$ is *bad* if **any** $h \in H$ is badly estimated. The probability of a union of events is at most the sum of their probabilities:

$$ P(D_{\text{train}}\text{ is bad}) = P\!\left( \bigcup_{h \in H} \{\, |L(h,D_{\text{train}}) - L(h,D_{\text{all}})| > \epsilon \,\} \right) \le \sum_{h \in H} 2\, e^{-2N\epsilon^2}. $$

The sum has $|H|$ identical terms, so

$$ \boxed{\,P(D_{\text{train}}\text{ is bad}) \;\le\; |H| \cdot 2\, e^{-2 N \epsilon^2}\,} $$

Read the three factors: **$|H|$** (more hypotheses → more ways to be fooled), **`N`** (more data → exponentially safer), **$\epsilon$** (looser tolerance → exponentially safer). Next we check this bound against the simulation.

In [ ]:
# Analytic Hoeffding + union bound, overlaid on Monte Carlo points.
def analytic_bound_fn(N, epsilon, H_size=10000):
    """P(bad) <= |H| * 2 exp(-2 N eps^2), capped at 1.0 (it is a probability). Scalar or array N."""
    return np.minimum(1.0, H_size * 2.0 * np.exp(-2.0 * np.asarray(N) * epsilon ** 2))

N_values = np.array([50, 100, 200, 300, 500, 750, 1000])
bound_curve = analytic_bound_fn(N_values, 0.1, 10000)
emp = np.array([monte_carlo_p_bad(int(N), 0.1, n_trials=400, seed=0) for N in N_values])

fig, ax = plt.subplots()
ax.plot(N_values, bound_curve, "o-", label="analytic bound |H|*2 exp(-2 N eps^2)")
ax.plot(N_values, np.maximum(emp, 0.5 / 400), "s--", label="Monte Carlo P(bad)")
ax.set_yscale("log")
ax.set_xlabel("N")
ax.set_ylabel("P(D_train is bad)")
ax.set_title("Hoeffding+union bound vs simulation (eps=0.1, |H|=10000)")
ax.legend()
bound_vs_empirical_figure = fig
plt.show()

for N in [100, 500, 1000]:
    print(f"N={N:>4}: bound={float(analytic_bound_fn(N, 0.1, 10000)):.4g}, "
          f"empirical={monte_carlo_p_bad(N, 0.1, 400, seed=0):.4g}")

assert analytic_bound_fn(100, 0.1, 10000) > analytic_bound_fn(1000, 0.1, 10000)  # decreases with N
assert float(analytic_bound_fn(1, 0.5, 10000)) == 1.0                            # capped at 1
for i in range(len(N_values)):  # bound holds within MC noise
    se = np.sqrt(max(emp[i], 1e-6) * (1 - emp[i]) / 400)
    assert emp[i] <= bound_curve[i] + 3 * se

In [ ]:
# Interactive bound explorer over N, epsilon, |H|.
def explore_bound(N, epsilon, H_size):
    N_axis = np.arange(20, 1001, 10)
    curve = analytic_bound_fn(N_axis, epsilon, H_size)
    cur = float(analytic_bound_fn(N, epsilon, H_size))
    fig, ax = plt.subplots()
    ax.plot(N_axis, curve, label=f"bound (eps={epsilon}, |H|={H_size})")
    ax.scatter([N], [cur], color="r", zorder=5)
    ax.axhline(cur, color="r", ls=":", alpha=0.5)
    ax.set_yscale("log")
    ax.set_xlabel("N")
    ax.set_ylabel("P(bad) bound")
    ax.set_title(f"N={N}, eps={epsilon}, |H|={H_size} -> bound={cur:.3g}")
    ax.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    bound_explorer_widget = interact(
        explore_bound,
        N=IntSlider(min=20, max=1000, step=10, value=100, continuous_update=False),
        epsilon=FloatSlider(min=0.02, max=0.30, step=0.01, value=0.1, continuous_update=False),
        H_size=SelectionSlider(options=[10, 100, 1000, 10000, 100000], value=10000),
    )
else:
    explore_bound(100, 0.1, 10000)  # static fallback
    bound_explorer_widget = None

## Turning the bound around: how much data do I need?

The bound answers a designer's question if we read it backwards. Demand a failure probability of at most $\delta$:

$$ |H| \cdot 2\, e^{-2 N \epsilon^2} \;\le\; \delta. $$

Solve for `N` (take natural logs):

$$ \boxed{\,N \;\ge\; \dfrac{\ln\!\big(2|H| / \delta\big)}{2\,\epsilon^2}\,} $$

This is the **sample complexity**: the number of i.i.d. examples that *guarantees* — with probability at least $1 - \delta$ — that training loss is within $\epsilon$ of ideal loss for **every** hypothesis. Notice `N` grows only **logarithmically in $|H|$** (cheap) but like **$1/\epsilon^2$** (expensive: halving the tolerance quadruples the data).

In [ ]:
# Sample complexity: invert the bound to N >= ln(2|H|/delta) / (2 eps^2).
def required_N_fn(epsilon, delta, H_size=10000):
    """Smallest N guaranteeing P(bad) <= delta. Natural log, to round-trip with C17's exp."""
    assert epsilon > 0 and 0 < delta < 1
    return int(np.ceil(np.log(2.0 * H_size / delta) / (2.0 * epsilon ** 2)))

N_req_slide = required_N_fn(0.1, 0.1, 10000)
print(f"Required N (|H|=10000, delta=0.1, eps=0.1): {N_req_slide}")

H_axis = np.logspace(1, 6, 50)
N_vs_H = np.log(2 * H_axis / 0.1) / (2 * 0.1 ** 2)
eps_axis = np.linspace(0.02, 0.3, 50)
N_vs_eps = np.log(2 * 10000 / 0.1) / (2 * eps_axis ** 2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(H_axis, N_vs_H, color="tab:blue")
ax1.set_xscale("log")
ax1.set_xlabel("|H|"); ax1.set_ylabel("required N")
ax1.set_title("N vs |H|  (delta=0.1, eps=0.1)")
ax1.scatter([10000], [N_req_slide], color="r", zorder=5, label=f"|H|=1e4 -> N={N_req_slide}")
ax1.legend()

ax2.plot(eps_axis, N_vs_eps, color="tab:green")
ax2.set_xlabel("epsilon"); ax2.set_ylabel("required N")
ax2.set_title("N vs epsilon  (|H|=10000, delta=0.1)")
ax2.scatter([0.1], [N_req_slide], color="r", zorder=5, label=f"eps=0.1 -> N={N_req_slide}")
ax2.legend()
sample_complexity_figure = fig
plt.tight_layout()
plt.show()

assert 605 <= N_req_slide <= 615, f"expected ~610, got {N_req_slide}"
assert required_N_fn(0.05, 0.1, 10000) > required_N_fn(0.1, 0.1, 10000)   # smaller eps needs more data
assert required_N_fn(0.1, 0.1, 100000) > required_N_fn(0.1, 0.1, 10000)   # bigger |H| needs more data

## The punchline: the $|H|$ tradeoff

Putting the pieces together exposes a genuine tension in choosing model complexity $|H|$:

- **Shrink $|H|$** → the bound $|H| \cdot 2 e^{-2N\epsilon^2}$ gets *smaller*, so the **generalization gap tightens**. But a smaller hypothesis set may not contain a good threshold, so the **best achievable loss $L(h_{\text{all}}, D_{\text{all}})$ rises**.
- **Grow $|H|$** → there is room for a better $h_{\text{all}}$, so **best-in-class loss falls** — but the **gap bound widens**, and a lucky sample can fool you.

What we ultimately pay is roughly **best-in-class loss + generalization gap**. Minimizing the *sum* means neither a tiny `H` (great gap, terrible fit) nor a huge `H` (great fit, terrible gap) is best — the sweet spot is in between. We compute it next, then explore it interactively.

In [ ]:
# The |H| tradeoff: best-in-class loss (down) vs generalization-gap term (up).
N_tradeoff, delta = 200, 0.1
H_sizes = np.array([2, 5, 10, 50, 100, 500, 1000, 5000, 10000])

realized_sizes, best_loss, gap, total = [], [], [], []
for k in H_sizes:
    # nested threshold subgrid; larger k strictly refines coverage of {1..10000}
    H_k = np.unique(np.linspace(1, 10000, k).round().astype(int))
    assert H_k.min() >= 1 and H_k.max() <= 10000
    best_k = float(L_all_over_H[H_k - 1].min())                              # decreasing in |H|
    gap_k = float(np.sqrt(np.log(2 * H_k.size / delta) / (2 * N_tradeoff)))  # increasing in |H|
    realized_sizes.append(int(H_k.size))
    best_loss.append(best_k)
    gap.append(gap_k)
    total.append(best_k + gap_k)

tradeoff_curves = {"H_sizes": realized_sizes, "best_loss": best_loss,
                   "gap": gap, "total": total, "N": N_tradeoff, "delta": delta}

xs = np.array(realized_sizes)
i_min = int(np.argmin(total))
fig, ax = plt.subplots()
ax.plot(xs, best_loss, "o-", label="best-in-class loss L(h_all, D_all)")
ax.plot(xs, gap, "s-", label=f"gap bound (N={N_tradeoff})")
ax.plot(xs, total, "^-", label="total = loss + gap")
ax.axvline(xs[i_min], color="k", ls=":")
ax.scatter([xs[i_min]], [total[i_min]], color="r", zorder=5, label=f"min total at |H|={xs[i_min]}")
ax.set_xscale("log")
ax.set_xlabel("|H|"); ax.set_ylabel("error")
ax.set_title("Complexity tradeoff: low loss wants big H, small gap wants small H")
ax.legend()
tradeoff_figure = fig
plt.show()

assert tradeoff_curves["best_loss"][-1] <= tradeoff_curves["best_loss"][0] + 1e-9  # loss decreases with |H|
assert tradeoff_curves["gap"][-1] >= tradeoff_curves["gap"][0]                     # gap increases with |H|
assert abs(tradeoff_curves["best_loss"][-1] - L_h_all) < 1e-9                      # full grid recovers L_h_all
assert len(tradeoff_curves["total"]) == len(tradeoff_curves["H_sizes"])

In [ ]:
# Interactive tradeoff explorer: find the |H| sweet spot for a chosen N.
def explore_tradeoff(H_size, N):
    delta = 0.1
    xs = np.array(tradeoff_curves["H_sizes"])
    best = np.array(tradeoff_curves["best_loss"])
    gap_at_N = np.sqrt(np.log(2 * xs / delta) / (2 * N))  # recompute from slider N (not C22's fixed N)
    total = best + gap_at_N
    k = int(np.argmin(np.abs(xs - H_size)))               # snap to nearest computed |H|
    fig, ax = plt.subplots()
    ax.plot(xs, best, "o-", label="best-in-class loss L(h_all,D_all)")
    ax.plot(xs, gap_at_N, "s-", label=f"gap bound (N={N})")
    ax.plot(xs, total, "^-", label="total = loss + gap")
    ax.set_xscale("log")
    ax.axvline(xs[k], color="k", ls=":")
    ax.scatter([xs[k]], [total[k]], color="r", zorder=5, label=f"|H|={xs[k]}: total={total[k]:.3f}")
    ax.set_xlabel("|H|"); ax.set_ylabel("error")
    ax.set_title("Move N and |H|: where is total error minimized?")
    ax.legend()
    plt.show()

if WIDGETS_AVAILABLE:
    _mid = tradeoff_curves["H_sizes"][len(tradeoff_curves["H_sizes"]) // 2]
    tradeoff_explorer_widget = interact(
        explore_tradeoff,
        H_size=SelectionSlider(options=list(tradeoff_curves["H_sizes"]), value=_mid),
        N=IntSlider(min=20, max=2000, step=20, value=200, continuous_update=False),
    )
else:
    _mid = tradeoff_curves["H_sizes"][len(tradeoff_curves["H_sizes"]) // 2]
    explore_tradeoff(_mid, 200)  # static fallback
    tradeoff_explorer_widget = None

## Wrap-up: the whole argument in one breath

We walked a single executable argument:

1. **Model → loss → optimization** (`C04`–`C09`): a 1-D threshold classifier on calibrated synthetic data, with $h_{\text{all}} \approx 4824$ and ideal loss $\approx 0.28$ found by scanning `H`.
2. **The sampling gap** (`C10`–`C12`): minimizing on a finite i.i.d. $D_{\text{train}}$ gives $h_{\text{train}}$, whose training loss is optimistic — sometimes wildly so on an unlucky sample.
3. **Good vs. bad sets** (`C13`–`C15`): a good sample estimates *every* hypothesis to within $\epsilon$, guaranteeing $2\epsilon$ generalization; we estimated $P(\text{bad})$ by Monte Carlo.
4. **The bound** (`C16`–`C18`): Hoeffding + union $\Rightarrow P(\text{bad}) \le |H| \cdot 2 e^{-2N\epsilon^2}$, verified to be loose-but-valid against the simulation.
5. **Sample complexity** (`C19`–`C20`): invert it to $N \ge \ln(2|H|/\delta) / (2\epsilon^2)$ — e.g. $N \approx 611$ for $|H|=10^4,\ \epsilon=\delta=0.1$.
6. **The tradeoff** (`C21`–`C23`): small $|H|$ tightens the gap but raises best-in-class loss; large $|H|$ does the reverse, so the total error has an interior minimum.

**The cliffhanger.** Can we have **both** a low ideal loss **and** a small generalization gap — *escape* the tradeoff instead of merely balancing it? The lecture's answer, and the next chapter of the story: **yes — that is what Deep Learning buys us.**